# The optical-depth integration: SkewerJAX against fake_spectra, one difference at a time

**Scope: the integration only.** Both codes are started from **identical input fields**: SkewerJAX's matched extraction, which reproduces fake_spectra's per-cell values (`3_fields_from_simulations_comparison_fake_spectra.ipynb`, §2), and fake_spectra's own per-cell values as a cross-check. Every remaining difference therefore comes from how τ is integrated.

**Plan.**
1. §2: SkewerJAX's default integration on those fields.
2. §3: fake_spectra's integration, reproduced by `ss.FAKE_SPECTRA_INTEGRATION`.
3. §4: every way fake_spectra's integration differs, one at a time, each with its matched mode and its measured cost on τ, F, ⟨F⟩ and the P1D.
4. §6: the two comparison notebooks combined. The total default − fake_spectra difference is decomposed into its extraction part (the fields notebook) and its integration part (this one), and the parts must add up.

**The default integration** is explained in `6_optical_depth_from_simulations.ipynb`.

**Runtime.** About 7 min after the fields notebook, whose extraction chain it reads from `CACHE` (about 15 min alone); under a minute fully cached.

## 0. Setup

In [ ]:
import os, sys, time, warnings
os.environ.setdefault("JAX_PLATFORMS", "cpu")             # float64 runs faster on this CPU than on a consumer GPU
import logging; logging.getLogger("jax._src.xla_bridge").setLevel(logging.CRITICAL)
from pathlib import Path
import numpy as np, jax, jax.numpy as jnp
jax.config.update("jax_enable_x64", True)
import matplotlib as mpl
import matplotlib.pyplot as plt
from skewerjax.config.setup_matplotlib import (matplotlib_tol_config, TOL_BRIGHT as TOL, ENTITY_COLORS as EC, MODE_STYLES as MS,
                                               tol_cmap, add_panel_letters, residual_band)
font, rcnew = matplotlib_tol_config()                         # the package style with Paul Tol's colour-blind-safe palette
plt.rc("font", **font); plt.rcParams.update(rcnew)

# ------------------------------------------------------------------ configuration (edit here)
REPO = Path("/home/dlopez/Documentos/0.profesional/Postdoc/USP/Projects/SkewerJAX")
DATA = REPO / "claude_agents/tmp_files/20261021_1335"        # caches and fake_spectra reference data (to be moved: see the notebook header)
SNAP_BASE, SNAPNUM = "/home/STORAGE/TNG50-4", 25          # read-only
CACHE = DATA / "cache" / "sjx_cache.h5"                    # SkewerJAX tau results (None: no cache)
DS_KMS = 0.25                                                # spacing of the sampling points, in Hubble velocity [km/s]
RES = 1.0                                                    # spectral pixel [km/s]
T_START = time.perf_counter()
from skewerjax.simulations import snapshot_readers as sr, gas_physics as gp, cell_selection as sel
from skewerjax.simulations import simulation_skewers as ss
from skewerjax.fields import deposition as dep
from skewerjax.optical_depth import absorbers as ab_
from skewerjax.optical_depth.optical_depth import OpticalDepthCalculator
from skewerjax.skewers.skewers import SkewerBundle, path_geometry, axis_aligned_skewer_points
from skewerjax.constants.lines import LineBundle
from skewerjax.constants.physical_constants import phys
print("jax", jax.__version__, jax.devices()[0], "| x64", jax.config.jax_enable_x64)

RESULTS = []
def check(name, value, tol, section, kind="pass", detail=""):
    """kind="pass": PASS if value <= tol; "explained": EXPLAINED if value <= tol (a difference accounted
    for by its parts); "expect": PASS if value > tol (the test must detect something)."""
    ok = bool(value > tol) if kind == "expect" else bool(value <= tol)
    tag = {"pass": ("PASS", "FAIL"), "expect": ("PASS", "FAIL"), "explained": ("EXPLAINED", "UNEXPLAINED")}[kind][0 if ok else 1]
    RESULTS.append(dict(section=section, name=name, value=float(value), tol=tol, kind=kind, tag=tag, detail=detail))
    print(f"[{tag}] {section:5s} {name}: {value:.3e} ({'>' if kind == 'expect' else '<='} {tol:g}) {detail}")
def rel(a, b, mask=None):
    a, b = np.asarray(a, float), np.asarray(b, float)
    m = np.ones(a.shape, bool) if mask is None else np.asarray(mask, bool)
    return np.abs(a[m] - b[m]) / np.maximum(np.abs(b[m]), 1e-300)

In [ ]:
# ------------------------------------------------------------------ fake_spectra: live (its build) or the reference file
from skewerjax.optical_depth.fake_spectra_reference import FakeSpectraSource, SKEWER_SETS
FAKE_SPECTRA_BUILD = REPO / "claude_agents/tmp_files/20261019_2155/fake_spectra_build"   # None: reference file only
FS_DRIVER = DATA / "run_fake_spectra_skewers.py"                                        # runs fake_spectra in its own process
FS_CACHE = DATA / "fs_cache"                                                             # its results (fs_*.h5, fs_products.h5)
FS_REFERENCE = DATA / "fake_spectra_reference_tng50-4_snap025.h5"                        # every fake_spectra number used here
EXPORT_REFERENCE = False                                                                  # True: add new fake_spectra results to FS_REFERENCE (live mode)
FS = FakeSpectraSource(FS_REFERENCE, build=FAKE_SPECTRA_BUILD, driver=FS_DRIVER, cache=FS_CACHE, snap_base=SNAP_BASE, snapnum=SNAPNUM, res=RES)
rd = sr.ArepoHDF5Reader(SNAP_BASE, SNAPNUM); H = rd.header; L = H.box_cMpc
FC = FS.constants()
LYA, LB_SI = LineBundle(["HI-LyA"], ref_line_id="HI-LyA"), LineBundle(["SiIII-1206"], ref_line_id="SiIII-1206")
SI = gp.Ion("Si", 3)
SK = {name: ss.SimulationSkewers.axis_aligned(rd, SKEWER_SETS[name]["cofm"], SKEWER_SETS[name]["axis"], DS_KMS, name=name, cache=CACHE)
      for name in ("single", "nset", "sk8")}
v, dv = SK["single"].spectral_grid(RES)                     # fake_spectra's periodic grid, shared by both codes
print("fake_spectra:", FS.mode, f"| spectral grid: {v.size} pixels of {dv:.6f} km/s")
for name, S in SK.items():
    print(f"{name:6s}: {S.n_skewers:2d} skewers x {S.points.shape[1]} points, {len(S.cells):,} cells selected in {S.selection_s:.1f} s")
E_FS, I_FS, E_DEF, I_DEF = ss.FAKE_SPECTRA_EXTRACTION, ss.FAKE_SPECTRA_INTEGRATION, ss.ExtractionOptions(), ss.IntegrationOptions()
EFFECT_STYLE = {                                             # one colour and line style per difference, in every figure
    "tautail": (TOL["grey"], "-"), "pixel average": (TOL["yellow"], ":"), "detuning": (TOL["cyan"], "-."), "7/8 normalisation": (TOL["green"], "-"),
    "chord quadrature": (TOL["yellow"], "--"), "frame": (TOL["purple"], "-"), "T constants": (TOL["cyan"], ":"), "b constants": (TOL["yellow"], "-."),
    "mass constant": (TOL["green"], "--"), "neutral fraction": (TOL["grey"], ":"), "CLOUDY density": (TOL["purple"], "--"),
    "CLOUDY interpolation": (TOL["red"], ":"), "deposition": (TOL["blue"], "-")}
def logbin(k, y, nb=40):
    """Mean of y in nb log-spaced k bins (figures only; every check uses all modes)."""
    e = np.geomspace(k[0], k[-1] * 1.0001, nb + 1); j = np.digitize(k, e) - 1; n = np.bincount(j, minlength=nb)[:nb]
    return np.sqrt(e[1:] * e[:-1])[n > 0], (np.bincount(j, weights=y, minlength=nb)[:nb] / np.maximum(n, 1))[n > 0]
FS_TAU = dict(single=FS.run("single_matched", "single", products="tau,colden,temp,velocity")["tau"], nset=FS.run("nset_default", "nset", sf_neutral=True)["tau"],
              sk8_SiIII=FS.run("nset_SiIII", "nset", species="Si:3:1206")["tau"][8:9])

## 1. Identical input fields

**The fields.** From here on the extraction is `FAKE_SPECTRA_EXTRACTION`: top-hat spheres, fake_spectra's mass and temperature constants, its line-width constants and its x_HI rule. Its per-cell values agree with fake_spectra's to ≤ 2×10⁻⁶ (rechecked below).

**The cross-check (same cells).** fake_spectra's own per-cell values are also fed to SkewerJAX, through `override` = `FS.cell_inputs(...)`: then the inputs are fake_spectra's, bit for bit, and only the integration engine is SkewerJAX's.

In [ ]:
S = SK["single"]; TH1 = S.segments("tophat"); touch = np.unique(TH1.cell[TH1.cell >= 0])
fi = FS.cell_inputs(rd, S.cells, "single", gp.HI)
P_fs = S.properties(gp.HI, E_FS)
for k, a_, b_ in (("n_HI", np.asarray(P_fs["n_ion"]), fi["n"]), ("T", np.asarray(P_fs["T"]), fi["T"]),
                  ("v_pec", np.asarray(S.cells.fields["velocity"]), fi["vel"]), ("size", np.asarray(S.cells.size_cMpc), fi["size"])):
    check(f"matched extraction == fake_spectra's per-cell {k} (cells touching the single skewer)", rel(a_[touch], b_[touch]).max(), 2e-6, "1")
t_same = S.tau(gp.HI, LYA, v, E_FS, I_FS, override=fi, override_key="fake_spectra_cells")
t_match = S.tau(gp.HI, LYA, v, E_FS, I_FS)
for lab, t_ in (("same cells (fake_spectra's per-cell values)", t_same), ("matched extraction", t_match)):
    m = ss.spectral_metrics(t_, FS_TAU["single"], dv)
    print(f"{lab:46s}: max |dF| {m['max_dF']:.1e}, d<F> {m['d_meanF']:+.1e}, dEW/EW {m['dEW']:+.1e}")
check("same cells + fake_spectra's integration == fake_spectra (max |dF|)", np.abs(np.exp(-t_same) - np.exp(-FS_TAU["single"])).max(), 1e-6, "1")

## 2. SkewerJAX's default integration on fake_spectra's fields

`IntegrationOptions()` on the matched fields: the exact frame a(x) along the path, every deposited piece with its exact length and its full profile, the exact detuning, the periodic minimum image, and pixel centres. The difference to fake_spectra below is the **whole integration difference**, which §4 takes apart.

In [ ]:
t_def1 = S.tau(gp.HI, LYA, v, E_FS, I_DEF)
m = ss.spectral_metrics(t_def1, FS_TAU["single"], dv)
print(f"single skewer, [fake_spectra fields, SkewerJAX integration] vs fake_spectra: max |dF| {m['max_dF']:.2e}, rms {m['rms_dF']:.2e}, "
      f"d<F> {m['d_meanF']:+.4f}, dEW/EW {m['dEW']:+.2%}, lag {m['lag']:+.1f} km/s")
def plot_pair(F_ref, Fs, window, title, ref_label="fake_spectra", tol=1e-5):
    """F against a reference, and the residual with the reproduction tolerance band (presentation only)."""
    fig, axs = plt.subplots(2, 1, figsize=(16, 10), sharex=True, layout="constrained", gridspec_kw={"height_ratios": [2.2, 1.6]})
    w = (v > window[0]) & (v < window[1])
    axs[0].plot(v[w], F_ref[w], color=EC["fake_spectra"], **MS["reference"], label=ref_label)
    for (lab, F), sty, c in zip(Fs.items(), (MS["default"], MS["matched"]), (EC["HI"], TOL["red"])):
        axs[0].plot(v[w], F[w], color=c, **sty, label=lab); axs[1].plot(v[w], (F - F_ref)[w], color=c, **sty)
    axs[0].set(ylabel="F", ylim=(-0.05, 1.12), title=title); axs[0].legend(loc="upper center", ncols=3, fontsize=14)
    if tol:
        residual_band(axs[1], tol, label=r"$\pm10^{-5}$: reproduction tolerance"); axs[1].legend(loc="lower right", fontsize=14)
    else:
        axs[1].axhline(0, color="k", lw=1)
    axs[1].set(xlabel="v [km/s]", ylabel=f"F − F({ref_label})" if len(ref_label) < 22 else "F − F(reference)"); axs[1].set_yscale("symlog", linthresh=1e-5)
    add_panel_letters(axs)
    plt.show()
Ffs1 = np.exp(-FS_TAU["single"][0])
j = np.argmax(np.abs(np.exp(-t_def1[0]) - Ffs1))
plot_pair(Ffs1, {"SkewerJAX default integration": np.exp(-t_def1[0]), "fake_spectra's integration (matched)": np.exp(-t_match[0])},
          (v[j] - 250, v[j] + 250), "Identical fields: SkewerJAX's integration vs fake_spectra's, reproduced (dashed)", tol=1e-5)

## 3. fake_spectra's integration, reproduced

fake_spectra computes, for every cell whose sphere touches the sightline (`absorption.cpp`, `singleabs.h`):
1. one line centre per cell, u_c = a H z + v_∥, with one a for the whole box;
2. the cell's chord, sampled at 7 interior points of 8 equal intervals, each of weight 1/8;
3. a Voigt profile linear in the detuning, H(a, Δu/b);
4. the pixel centre if the pixel is narrower than b/2, else a trapezoid average over the pixel;
5. the nearest periodic image, walking out from u_c and stopping at the first pixel that receives less than `tautail` = 10⁻⁷ from that cell.

`FAKE_SPECTRA_INTEGRATION` = `IntegrationOptions(frame="snapshot", chord="fake_7pt", detuning="linear", tautail=True, pixel_average=True)` re-implements each of these steps on SkewerJAX's absorbers (`optical_depth/fake_spectra_emulation.py`). On the matched fields it must reproduce fake_spectra.

In [ ]:
t_match_n = SK["nset"].tau(gp.HI, LYA, v, E_FS, I_FS)
for name, t_, ref in (("single", t_match, FS_TAU["single"]), ("nset", t_match_n, FS_TAU["nset"])):
    check(f"{name}: [matched fields, fake_spectra integration] == fake_spectra, max |dF|", np.abs(np.exp(-t_) - np.exp(-ref)).max(), 1e-5, "3")
    mm = ref > 1e-5
    check(f"{name}: max |dtau|/tau (tau > 1e-5, saturated pixels included)", rel(t_, ref, mm).max(), 1e-4, "3")
k_, P_m = ss.p1d(t_match_n, dv); _, P_f = ss.p1d(FS_TAU["nset"], dv)
band = (k_ > 2e-3) & (k_ < 1.0)
check("nset: P1D == fake_spectra (k < 1 s/km), max |dP/P|", np.abs(P_m / P_f - 1)[band].max(), 1e-3, "3")

## 4. Each difference of the integration, one at a time

The differences are switched from fake_spectra's integration to SkewerJAX's in the order below, on the matched fields (`INTEGRATION_EFFECTS`).
- **Chain:** each step applied on top of the previous ones. It telescopes exactly from fake_spectra's integration to SkewerJAX's.
- **Isolated:** each step applied alone, from fake_spectra's integration. The frame needs the exact chord, so its isolated switch starts from the step "chord quadrature".

| step | fake_spectra | SkewerJAX | matched mode |
|---|---|---|---|
| tautail | each cell's profile stops at τ < 10⁻⁷ | every profile in full | `tautail=True` |
| pixel average | per-cell trapezoid over the pixel if b < 2Δv | pixel centres | `pixel_average=True` |
| detuning | H(a, Δu/b), linear in Δu | ν = (c/λ) expm1(−Δu/c) | `detuning="linear"` |
| 7/8 normalisation | 7 chord points of weight 1/8: 7/8 of the column | the whole column | `normalisation=1` with `fake_7pt` |
| chord quadrature | 7 points per chord | the exact chord, in pieces | `chord="fake_7pt"` |
| frame | one a; u = aH z + v_pec (linear Hubble flow) | a(x) along the path; Λ = c ln(a_†/a) + Doppler | `frame="snapshot"` |

Both codes use the periodic nearest image, so the wrap is not a step of the chain; §4.1 shows what it does.

In [ ]:
RI = {name: ss.run_effects(SK[name], gp.HI, LYA, v, ss.INTEGRATION_EFFECTS, E_FS, I_FS) for name in ("single", "nset")}
ROWS_I, TOT_I = ss.effect_rows(RI["nset"], dv, FS_TAU["nset"])
ss.print_effect_rows(ROWS_I, TOT_I, "\nLyα, 16 skewers: the integration differences (matched fields)")
x_end, i_end = RI["nset"]["options"][list(RI["nset"]["links"])[-1]]
check("integration chain ends at [matched fields, SkewerJAX integration]", float(i_end != I_DEF), 0, "4")
check("integration total = sum of the isolated differences, d<F>", abs(TOT_I["dF"] - TOT_I["dF_sum"]), 0.1 * TOT_I["dF_abs"] + 1e-6, "4", kind="explained")
check("integration total = sum of the isolated differences, dEW/EW", abs(TOT_I["dEW"] - TOT_I["dEW_sum"]), 0.1 * TOT_I["dEW_abs"] + 1e-6, "4", kind="explained")
L1 = RI["single"]["links"]; LAB = list(L1)
def step_pair(a, b, title, window=None, k=0):
    Fa, Fb = np.exp(-L1[a][k]), np.exp(-L1[b][k])
    j = np.argmax(np.abs(Fb - Fa)); window = window or (v[j] - 200, v[j] + 200)
    plot_pair(Fa, {f"after the step: {b}": Fb}, window, title, ref_label="before the step", tol=None)
    r_ = next(r for r in ROWS_I if r["effect"] == b)
    print(f"{b}: single skewer max |dF| {np.abs(Fb - Fa).max():.2e} | 16 skewers (isolated): rms dF {r_['rms_iso']:.2e}, max {r_['max_iso']:.2e}, "
          f"d<F> {r_['dF_iso']:+.2e}, dEW/EW {r_['dEW_iso']:+.2e}, P1D |dP/P| {r_['p1d_iso'][0]:.1e} (k < 0.1), {r_['p1d_iso'][1]:.1e} (0.1-1 s/km)")

### 4.1 The periodic wrap (both codes)

**How the two relate.** A periodic box has a periodic spectrum. The exact periodic spectrum is the sum over all images of every absorber, Σ_k φ(u − u_c + kP), with P the box length in velocity. fake_spectra keeps only the **nearest image** of each absorber: its profile is added within ±P/2 of its centre, with pixel indices wrapped mod n_bins. So "periodic wrap" and "one image truncated at half a box" are the same rule.

**SkewerJAX's equivalent** is the minimum-image detuning (`periodic=True`, the default, which sets `velocity_period = P`). It is exactly fake_spectra's rule, so the matched variant reproduces fake_spectra also at the two ends of the box (checked below).

**Alternatives,** with SkewerJAX's own integral in fake_spectra's frame (step "chord quadrature" of the chain, where the frame is still the snapshot's):
- **(ii) extended sampling:** the skewer extended by L/2 beyond each end, integrated directly (`periodic=False`) and evaluated on [0, P). It includes every image whose gas lies on the extended path.
- **(iii) the direct integral** of the box alone: what a finite sightline would give.
- **(iv) three images:** extended by L on each side, i.e. the sum over the images k = −1, 0, +1. It measures what the truncation at half a box drops, on skewer 8 (the damped absorber).

In [ ]:
I_STAR = RI["single"]["options"]["chord quadrature"][1]                    # exact chord, snapshot frame, periodic
edge = (v < 150) | (v > H.vmax_kms - 150)
F_i = np.exp(-L1["chord quadrature"][0])
check("(i) the matched wrap reproduces fake_spectra at the ends of the box", np.abs(np.exp(-t_match[0]) - Ffs1)[edge].max(), 1e-5, "4.1")
t_iii = S.tau(gp.HI, LYA, v, E_FS, I_STAR.replace(periodic=False))[0]
ext = lambda name, cofm, axis, e: ss.SimulationSkewers.axis_aligned(rd, cofm, axis, DS_KMS, extend_cMpc=e, name=name, cache=CACHE)
S_ii = ext("single_ext_half", SKEWER_SETS["single"]["cofm"], SKEWER_SETS["single"]["axis"], L / 2)
t_ii = S_ii.tau(gp.HI, LYA, v, E_FS, I_STAR.replace(periodic=False))[0]
for lab, t_ in (("(ii) extended by L/2", t_ii), ("(iii) direct, box only", t_iii)):
    d_ = np.exp(-t_) - F_i
    print(f"{lab:24s} vs (i): max |dF| within 150 km/s of an end {np.abs(d_)[edge].max():.2e}, elsewhere {np.abs(d_)[~edge].max():.2e}")
check("(iii) without the wrap the ends lose the gas across the faces", np.abs(np.exp(-t_iii) - F_i)[edge].max(), 1e-2, "4.1", kind="expect")
S8x = ext("sk8_ext_full", SKEWER_SETS["sk8"]["cofm"], SKEWER_SETS["sk8"]["axis"], L)
t8_one = SK["sk8"].tau(gp.HI, LYA, v, E_FS, I_STAR)[0]
t8_three = S8x.tau(gp.HI, LYA, v, E_FS, I_STAR.replace(periodic=False))[0]
dF8 = np.exp(-t8_three) - np.exp(-t8_one); jd = np.argmax(t8_one)
far = np.abs(np.mod(v - v[jd] + H.vmax_kms / 2, H.vmax_kms) - H.vmax_kms / 2) > 1000
print(f"skewer 8: three images vs nearest image: max |dF| {np.abs(dF8).max():.2e}; more than 1000 km/s from the DLA {np.abs(dF8[far]).max():.2e}")
fig, axs = plt.subplots(1, 3, figsize=(22, 7), layout="constrained")
for ax_, (lo, hi), ttl in ((axs[0], (0, 200), "Near end: without the wrap the gas across the face is lost"), (axs[1], (H.vmax_kms - 200, H.vmax_kms), "Far end")):
    w = (v > lo) & (v < hi)
    for lab, t_, c, sty in (("(i) nearest image (both codes)", L1["chord quadrature"][0], EC["HI"], MS["default"]), ("(ii) extended by L/2", t_ii, TOL["green"], MS["matched"]),
                            ("(iii) direct, box only", t_iii, TOL["red"], MS["alternative"])):
        ax_.plot(v[w], np.exp(-t_)[w], color=c, **sty, label=lab)
    ax_.set(xlabel="v [km/s]", ylim=(-0.05, 1.12)); ax_.set_title(ttl, fontsize=17)
axs[0].set_ylabel("F"); axs[0].legend(loc="upper right", fontsize=14)
axs[2].plot(v, dF8, color=EC["HI"], lw=1.5); axs[2].axvline(v[jd], color=TOL["red"], ls="--", lw=2); axs[2].axhline(0, color="k", lw=1)
axs[2].text(v[jd] + 60, dF8.min() * 0.95, "DLA", color=TOL["red"], fontsize=15)
axs[2].set(xlabel="v [km/s]", ylabel="F(3 images) − F(nearest)"); axs[2].set_title("Skewer 8: the truncation drops < 10⁻² everywhere", fontsize=17)
add_panel_letters(axs)
plt.show()

*Figure — the ends of the single skewer, and the truncation on the damped absorber.*
- **(i)** and **(ii)** agree near the ends: the nearest image is the periodic spectrum.
- **(iii)**, the direct integral, loses the absorption of the gas across the faces. That is the right answer for a finite sightline, but not for a periodic box.
- **(iv)**, right panel: the truncation at half a box drops the damping wings of the neighbouring images of the damped absorber. They are small and spread over the whole box.

**Cost:** zero between the two codes, which share the rule. For SkewerJAX: none, since the minimum image costs one `mod`.

### 4.2 The early stop `tautail`

fake_spectra stops walking out of each cell's profile at the first pixel that receives less than 10⁻⁷, cutting the far Lorentzian wings cell by cell. SkewerJAX adds every profile in full.

In [ ]:
step_pair("start", "tautail", "tautail: fake_spectra's cut (reference) vs every profile in full")
k_, P_tt = ss.p1d(RI["nset"]["isolated"]["tautail"], dv); _, P_0 = ss.p1d(RI["nset"]["links"]["start"], dv)
hk = k_ > 0.8
print(f"P1D above k = 0.8 s/km, full profiles vs tautail: median dP/P {np.median(P_tt[hk] / P_0[hk] - 1):+.2f} "
      f"(fake_spectra's cut adds a white-noise floor kP/pi ~ {np.median(k_[hk] * P_0[hk] / np.pi):.1e})")

**Cost.** On F it is negligible: ~10⁻⁵, in the low-τ gaps where the summed far wings of many cells are all that is left. On the P1D at k ≳ 0.8 s/km it dominates: each abrupt cut at 10⁻⁷ adds white noise, which exceeds the thermally suppressed Lyα power there.

### 4.3 The pixel average

When a cell's line is narrower than twice the pixel (b < 2Δv), fake_spectra averages its profile over the pixel (a trapezoid on 2⌈Δv/b⌉+1 points); otherwise it takes the pixel centre. SkewerJAX evaluates the spectral grid where it is asked.

In [ ]:
b_cells = np.asarray(gp.doppler_b_kms(S.properties(gp.HI, E_FS)["T"], 1.00794))
print(f"Lyα: cells touching the skewer with b < 2 dv (averaged by fake_spectra): {np.mean(b_cells[touch] < 2 * dv):.2%}")
step_pair("tautail", "pixel average", "pixel average (reference) vs pixel centres")

**Cost.** It acts only on lines with b < 2Δv: very cold gas, metal lines, or coarse pixels. For Lyα at 1 km/s pixels that is almost no cell, so this step is negligible; for Si III on skewer 8 it is too (§7), because its cells are warmer than ~7000 K. With coarser pixels it grows.

### 4.4 The detuning

fake_spectra evaluates H(a, Δu/b), linear in Δu = u − u_c. The exact frequency offset is ν = (c/λ) expm1(−Δu/c), which differs at relative order Δu/(2c): 7×10⁻⁵ at 40 km/s. The exact mapping makes each profile slightly asymmetric, wider on the red side.

In [ ]:
step_pair("pixel average", "detuning", "linear detuning (reference) vs the exact frequency offset")

**Cost:** ≲ 3×10⁻⁴ in F, largest on the steep flanks of saturated lines.

### 4.5 The 7/8 normalisation

fake_spectra samples each cell's chord at the 7 *interior* points i = 1…7 of 8 intervals, each of weight 1/8. The two end points are skipped: they carry zero weight for the smoothing kernels the code was written for, which vanish at the edge. A top hat, however, is constant up to its edge. **fake_spectra's τ therefore integrates 7/8 of each cell's column**, while its own column density integrates all of it. This step keeps the 7 points and gives them weight 1/7 (`normalisation=8/7`).

In [ ]:
step_pair("detuning", "7/8 normalisation", "the 7/8 normalisation (reference) vs the full column (weights 1/7)")
t_a, t_b = L1["detuning"][0], L1["7/8 normalisation"][0]
check("7/8: tau ratio is exactly 8/7", np.abs(t_b[t_a > 1e-8] / t_a[t_a > 1e-8] - 8 / 7).max(), 1e-10, "4.5")

**Cost.** τ is 12.5% lower everywhere in fake_spectra. This is the largest single effect on ⟨F⟩ and on the equivalent widths (table above). It is a quadrature artefact, inconsistent with fake_spectra's own column densities.

### 4.6 The chord quadrature

With the full column restored, the remaining difference is the sampling itself. The 7 points are spaced aH·2z_r/8, up to ~10 km/s on TNG50-4's large cells. SkewerJAX integrates the exact chord of each sphere in each dual segment, in pieces of ≤ 0.25 km/s.

In [ ]:
step_pair("7/8 normalisation", "chord quadrature", "fake_spectra's 7 chord points (reference) vs the exact chord")

**Cost.** It changes line *shapes* by up to a few 10⁻² in F, as much as the 7/8 per pixel, but it changes ⟨F⟩ far less, and it acts mostly at high k in the P1D.

### 4.7 The frame: one redshift and a linear Hubble flow, against a(x)

**fake_spectra.** One a, one H(z) and one velfac for the whole box, so u = aH z + v_pec: a linear velocity mapping.

**SkewerJAX.** The box sits between the emitter and the observer, with a(x) from the cosmology (a = a_snap at the box centre). Each piece absorbs at Λ_c = c ln(a_†/a(x)) plus the relativistic Doppler shift of its v_pec. This differs from the linear map quadratically along the box (−2.9 km/s at the centre, zero at the ends). The comoving density is also conserved along the path, so the weight of the same gas, ∝ a⁻², changes by ±1.3% from one end to the other.

**Matched mode** (`frame="snapshot"`): SkewerJAX's line centre is made exactly u by injecting the velocity c tanh(u/c) into the relativistic Doppler factor, at a = a_snap everywhere.

In [ ]:
fe = S.frame("exact"); s1 = S.s[0]
u_ex = phys.c_kms * np.log(fe.a_dagger / fe.a_grid[0])
print(f"exact frame: P = {fe.period_kms:.4f} km/s (linear {H.vmax_kms:.4f}); max |Lambda_c - a H s| = {np.abs(u_ex - H.velfac_kms_per_cMpc * s1).max():.2f} km/s; "
      f"column factor (a_snap/a)^2 from {(H.a / fe.a_grid[0, 0]) ** 2:.4f} to {(H.a / fe.a_grid[0, -1]) ** 2:.4f}")
step_pair("chord quadrature", "frame", "one a and a linear Hubble flow (reference) vs a(x) along the path")
k_, P_ex = ss.p1d(RI["nset"]["links"]["frame"], dv); _, P_lin = ss.p1d(RI["nset"]["links"]["chord quadrature"], dv)
fig, axs = plt.subplots(1, 2, figsize=(20, 6.5), layout="constrained")
axs[0].plot(s1 * H.velfac_kms_per_cMpc, u_ex - H.velfac_kms_per_cMpc * s1, color=EFFECT_STYLE["frame"][0], lw=2.5); axs[0].axhline(0, color="k", lw=1)
axs[0].set(xlabel="a H s [km/s]", ylabel=r"$\Lambda_c - aHs$ [km/s]", title="Exact minus linear line centre: −2.9 km/s at the centre")
kb_, r_ex = logbin(k_, P_ex); _, r_lin = logbin(k_, P_lin)
axs[1].semilogx(kb_, (r_ex / r_lin - 1) * 100, "o-", color=EFFECT_STYLE["frame"][0], lw=2, ms=6); axs[1].axhline(0, color="k", lw=1)
axs[1].set(xlabel="k [s/km]", ylabel="ΔP/P [%] (40 log bins)", title="The frame on the P1D: ~1% at k < 0.1 s/km")
add_panel_letters(axs)
plt.show()

**Cost.** A position-dependent shift of the lines by a fraction of their width, plus the ±1.3% change of their strength. Per pixel this reaches 0.19 in F on the steep flanks of saturated lines (rms 0.02), but ⟨F⟩ changes by only 10⁻⁵, because the shifts and the strength changes nearly cancel over the box. On the P1D it is 1% at k < 0.1 s/km and 11% at 0.1–1 s/km.

**When each is appropriate.** For a periodic box, one epoch is self-consistent: all the gas is at one time. a(x) is exact for a light cone, and it makes Λ the real log λ. SkewerJAX defaults to a(x) because that is the integral it computes; `frame="snapshot"` reproduces fake_spectra's choice.

### 4.8 The Voigt function

fake_spectra uses a first-order approximation in the damping parameter for a ≤ 10⁻³ and the Faddeeva function otherwise. SkewerJAX uses ExoJAX's `voigtone`. Both are tested against Re w(u + ia) of `scipy.special.wofz`.

In [ ]:
from scipy.special import wofz
from skewerjax.optical_depth.voigt import log10_voigt
uu = np.linspace(0, 60, 6001)
for lab, aa in (("Lyα, a = 4.7e-4", 4.7e-4), ("Si III, a = 9.8e-3", 9.8e-3)):
    Hx = wofz(uu + 1j * aa).real
    H_exo = 10 ** np.asarray(log10_voigt(jnp.asarray(uu), 1 / np.sqrt(2), aa)) * np.sqrt(np.pi)
    check(f"ExoJAX voigtone vs wofz, {lab}, u <= 60", rel(H_exo, Hx).max(), 1e-5, "4.8")

**Cost:** < 10⁻⁶, below every other difference; not a step of the chain.

## 5. The integration differences, ranked

In [ ]:
ss.print_effect_rows(ROWS_I, TOT_I, "Lyα, 16 skewers, identical (matched) fields: ranked by the rms dF of the isolated switch")
P_r = {lab: ss.p1d(t_, dv)[1] for lab, t_ in RI["nset"]["links"].items()}
fig, ax = plt.subplots(figsize=(16, 7.5), layout="constrained")
for r_ in ROWS_I:
    b_ = RI["nset"]["base"][r_["effect"]]
    d_ = ss.p1d(RI["nset"]["isolated"][r_["effect"]], dv)[1] / P_r[b_] - 1
    c, ls = EFFECT_STYLE[r_["effect"]]
    kb_, Pi = logbin(k_, ss.p1d(RI["nset"]["isolated"][r_["effect"]], dv)[1]); _, Pb_ = logbin(k_, P_r[b_])
    ax.semilogx(kb_, Pi / Pb_ - 1, color=c, ls=ls, lw=2.5, marker="o", ms=4, label=r_["effect"])
ax.set_yscale("symlog", linthresh=1e-4); ax.axhline(0, color="k", lw=1)
ax.annotate("tautail: white-noise floor", xy=(1.2, -1), xytext=(0.15, -0.05), fontsize=15, arrowprops=dict(arrowstyle="->"))
ax.set(xlabel="k [s/km]", ylabel="ΔP/P (isolated, 40 log bins)", title=r"Ly$\alpha$ P1D: the 7/8 acts at all k, line shapes at high k, tautail above 0.8 s/km")
ax.legend(loc="upper left", ncols=3, fontsize=14); plt.show()

## 6. Both notebooks combined: the total difference, decomposed

The total difference between SkewerJAX's default and fake_spectra splits exactly into three parts, with I the integration and E the extraction:

$$F[E_{\rm def}, I_{\rm def}]-F_{\rm fake}=\underbrace{F[E_{\rm fs}, I_{\rm fs}]-F_{\rm fake}}_{\text{reproduction residual}}+\underbrace{F[E_{\rm fs}, I_{\rm def}]-F[E_{\rm fs}, I_{\rm fs}]}_{\text{integration (this notebook)}}+\underbrace{F[E_{\rm def}, I_{\rm def}]-F[E_{\rm fs}, I_{\rm def}]}_{\text{extraction (fields notebook)}}$$

Each part is in turn the sum of its chain steps (exact), and approximately the sum of its isolated differences: the effects interact through F = e^(−τ). The extraction chain is that of the fields notebook (§4 there), recomputed here or read from `CACHE`.

**Checks:**
- the telescoping must hold exactly;
- each part must equal the sum of its isolated differences within 10% of their summed magnitudes;
- so must the total (assessed as EXPLAINED).

In [ ]:
RE = ss.run_effects(SK["nset"], gp.HI, LYA, v, ss.EXTRACTION_EFFECTS, E_FS, I_DEF)
ROWS_E, TOT_E = ss.effect_rows(RE, dv, FS_TAU["nset"])
t_default = RE["links"][list(RE["links"])[-1]]
check("the default of the chain == SkewerJAX's default tau", np.abs(t_default - SK["nset"].tau(gp.HI, LYA, v)).max(), 0, "6")
mF = lambda t_: np.exp(-t_).mean(); EW = lambda t_: ab_.equivalent_width_kms(t_, dv).sum(); EWf = EW(FS_TAU["nset"])
res = dict(dF=mF(RI["nset"]["links"]["start"]) - mF(FS_TAU["nset"]), dEW=(EW(RI["nset"]["links"]["start"]) - EWf) / EWf)
total = dict(dF=mF(t_default) - mF(FS_TAU["nset"]), dEW=(EW(t_default) - EWf) / EWf)
for q, lab in (("dF", "d<F>"), ("dEW", "dEW/EW")):
    parts = res[q] + TOT_I[q] + TOT_E[q]
    iso = res[q] + TOT_I[f"{q}_sum"] + TOT_E[f"{q}_sum"]
    print(f"{lab:7s} total {total[q]:+.5e} = residual {res[q]:+.1e} + integration {TOT_I[q]:+.5e} + extraction {TOT_E[q]:+.5e} "
          f"| isolated: integration {TOT_I[f'{q}_sum']:+.5e}, extraction {TOT_E[f'{q}_sum']:+.5e}, all {iso:+.5e}")
    check(f"{lab}: total = residual + integration + extraction (telescoping)", abs(total[q] - parts), 1e-12, "6")
    check(f"{lab}: total = residual + sum of all isolated differences", abs(total[q] - iso), 0.1 * (TOT_I[f"{q}_abs"] + TOT_E[f"{q}_abs"]) + 1e-6, "6", kind="explained")
k_, P_def = ss.p1d(t_default, dv); _, P_fake = ss.p1d(FS_TAU["nset"], dv)
tot_P = P_def / P_fake - 1
sum_P = P_m / P_fake - 1
for R_, rows_ in ((RI["nset"], ROWS_I), (RE, ROWS_E)):
    PB = {lab: ss.p1d(t_, dv)[1] for lab, t_ in R_["links"].items()}
    for r_ in rows_:
        sum_P = sum_P + ss.p1d(R_["isolated"][r_["effect"]], dv)[1] / PB[R_["base"][r_["effect"]]] - 1
lo = (k_ > 2e-3) & (k_ < 0.1)
check("P1D (k < 0.1 s/km): total = sum of all isolated differences (max deviation / max total)",
      np.abs(tot_P - sum_P)[lo].max() / np.abs(tot_P[lo]).max(), 0.3, "6", kind="explained")
fig, ax = plt.subplots(figsize=(16, 7.5), layout="constrained")
kb_, Pd_ = logbin(k_, P_def); _, Pf_ = logbin(k_, P_fake); _, sP_ = logbin(k_, sum_P)
ax.semilogx(kb_, Pd_ / Pf_ - 1, color=EC["fake_spectra"], **MS["reference"], label="total: default / fake_spectra − 1")
ax.semilogx(kb_, sP_, color=EC["HI"], **MS["matched"], marker="o", ms=5, label="residual + sum of all isolated differences")
ax.axvspan(k_[0], 0.1, color=TOL["yellow"], alpha=0.15, lw=0); ax.text(2.2e-3, 2e-3, "checked range:\nthe parts add up", fontsize=15)
ax.set_yscale("symlog", linthresh=1e-3); ax.axhline(0, color="k", lw=1); ax.set_ylim(-2, 20)
ax.set(xlabel="k [s/km]", ylabel="ΔP/P (40 log bins)", title=r"Ly$\alpha$ P1D: the parts add up at k < 0.1 s/km; at higher k the effects interact")
ax.legend(loc="lower left"); plt.show()
print("\nall differences, ranked by the rms dF of the isolated switch (Lyα, 16 skewers):")
print(f"{'rank':4s} {'difference':22s} {'part':11s} {'rms dF':>9s} {'max |dF|':>9s} {'d<F>':>10s} {'dEW/EW':>10s} {'P1D k<0.1':>10s} {'0.1-1':>9s}")
allr = [(r_, "integration") for r_ in ROWS_I] + [(r_, "extraction") for r_ in ROWS_E]
for i, (r_, part) in enumerate(sorted(allr, key=lambda x: -x[0]["rms_iso"]), 1):
    print(f"{i:4d} {r_['effect']:22s} {part:11s} {r_['rms_iso']:9.2e} {r_['max_iso']:9.2e} {r_['dF_iso']:10.2e} {r_['dEW_iso']:10.2e} {r_['p1d_iso'][0]:10.2e} {r_['p1d_iso'][1]:9.2e}")

*Figure — the P1D.* At k < 0.1 s/km the sum of the isolated differences tracks the total (6% of its maximum, checked above). At higher k the effects interact: one changes the line shapes that another then moves, so the cross terms are not small there. Above k ≈ 0.8 s/km the total is −1: fake_spectra's power there is the white-noise floor of `tautail` (§4.2), which SkewerJAX does not have, and relative changes from different bases no longer add. The chains are the exact decomposition, because they telescope.

## 7. A metal line: Si III on skewer 8

The same integration chain, for Si III 1206.5 Å on skewer 8 (the only strong Si III system in this box). Its lines are 5× narrower than Lyα's, so the steps that move or reshape lines weigh more relative to their effect on Lyα.

In [ ]:
S8 = SK["sk8"]
t8_match = S8.tau(SI, LB_SI, v, E_FS, I_FS)
check("Si III: [matched fields, fake_spectra integration] == fake_spectra, max |dF|", np.abs(np.exp(-t8_match) - np.exp(-FS_TAU["sk8_SiIII"])).max(), 1e-5, "7")
RI8 = ss.run_effects(S8, SI, LB_SI, v, ss.INTEGRATION_EFFECTS, E_FS, I_FS)
ROWS_8, TOT_8 = ss.effect_rows(RI8, dv, FS_TAU["sk8_SiIII"])
ss.print_effect_rows(ROWS_8, TOT_8, "\nSi III, skewer 8: the integration differences (matched fields)")
RE8 = ss.run_effects(S8, SI, LB_SI, v, ss.EXTRACTION_EFFECTS, E_FS, I_DEF)
ROWS_8E, TOT_8E = ss.effect_rows(RE8, dv, FS_TAU["sk8_SiIII"])
t8_def = RE8["links"][list(RE8["links"])[-1]]
tot8 = (EW(t8_def) - EW(FS_TAU["sk8_SiIII"])) / EW(FS_TAU["sk8_SiIII"])
res8 = (EW(t8_match) - EW(FS_TAU["sk8_SiIII"])) / EW(FS_TAU["sk8_SiIII"])
print(f"Si III EW: default - fake_spectra {tot8:+.2%} = residual {res8:+.1e} + integration {TOT_8['dEW']:+.2%} + extraction {TOT_8E['dEW']:+.2%}")
check("Si III dEW/EW: total = residual + integration + extraction (telescoping)", abs(tot8 - res8 - TOT_8["dEW"] - TOT_8E["dEW"]), 1e-12, "7")
check("Si III dEW/EW: total = residual + sum of all isolated differences", abs(tot8 - res8 - TOT_8["dEW_sum"] - TOT_8E["dEW_sum"]),
      0.1 * (TOT_8["dEW_abs"] + TOT_8E["dEW_abs"]) + 1e-6, "7", kind="explained")
j = np.argmax(FS_TAU["sk8_SiIII"][0]); w8 = np.abs(v - v[j]) < 120
fig, ax = plt.subplots(figsize=(16, 7.5), layout="constrained")
ax.plot(v[w8], np.exp(-FS_TAU["sk8_SiIII"][0][w8]), color=EC["fake_spectra"], **MS["reference"], label="fake_spectra")
ax.plot(v[w8], np.exp(-RI8["links"]["start"][0][w8]), color=EC["SiIII"], **MS["matched"], label="fake_spectra's integration (matched)")
for lab in ("7/8 normalisation", "frame"):
    c, ls = EFFECT_STYLE[lab]
    ax.plot(v[w8], np.exp(-RI8["links"][lab][0][w8]), color=c, ls=ls if ls != "-" else "-.", lw=2, label=f"chain up to: {lab}")
ax.plot(v[w8], np.exp(-t8_def[0][w8]), color=EC["SiIII"], **MS["default"], label="SkewerJAX default (+ extraction)")
ax.set(xlabel="v [km/s]", ylabel="F", ylim=(-0.05, 1.12), title="Si III, skewer 8: the frame moves the edges, the 7/8 deepens the trough")
ax.legend(loc="lower left", fontsize=14); plt.show()
if EXPORT_REFERENCE and FS.live:
    print(f"reference file {FS_REFERENCE.name}: {FS.export() / 1e6:.1f} MB")

*Figure — Si III.* The narrow, saturated trough shows the steps that move lines (the frame) and those that change the column (the 7/8 normalisation) much more sharply than Lyα: a shift of a few km/s moves its edges by a large fraction of their width. The pixel average and `tautail` do nothing here: the cells are warmer than the averaging threshold, and the line is strong.

## 8. Summary of the checks

In [ ]:
import collections, resource
print(f"{'tag':12s} {'section':7s} name")
for r in RESULTS:
    print(f"{r['tag']:12s} {r['section']:7s} {r['name']}  ({r['value']:.2e} {'>' if r['kind'] == 'expect' else '<='} {r['tol']:g}) {r['detail']}")
print("\nSUMMARY:", dict(collections.Counter(r["tag"] for r in RESULTS)), f"of {len(RESULTS)} checks")
print(f"wall time {time.perf_counter() - T_START:.0f} s | peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")